# Initialization

In [2]:
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [2]:
%matplotlib inline
%config InlineBackend.figure_format = 'png'
%config InlineBackend.figure_format = 'retina'

# Загрузка данных

In [5]:
# старые данные
books = pd.read_parquet("goodreads/books.parquet")
interactions = pd.read_parquet("goodreads/interactions.parquet")

In [3]:
items = pd.read_parquet("items.par")
events = pd.read_parquet("events.par")

In [10]:
print(items.shape)
items.describe()

(43312, 19)


,item_id,num_pages,average_rating,ratings_count,text_reviews_count,publication_year
count,4.331200e+04,37001.0,43312.00000,43312.0,43312.000000,35891.0
mean,8.050797e+06,337.866085,3.99872,13361.85272,637.829239,2006.881084
std,9.315783e+06,256.097045,0.31284,78911.120232,2553.203776,167.257929
min,1.000000e+00,0.0,0.00000,0.0,0.000000,13.0
25%,2.544730e+05,224.0,3.81000,328.0,29.000000,2002.0
50%,3.354695e+06,313.0,4.01000,1834.0,120.000000,2008.0
75%,1.429010e+07,400.0,4.20000,7310.0,435.000000,2012.0
max,3.652450e+07,14777.0,5.00000,4899965.0,142645.000000,20136.0


In [11]:
print(events.shape)
events.describe()

(11751086, 7)


,user_id,item_id,rating
count,1.175109e+07,1.175109e+07,1.175109e+07
mean,1.215043e+06,9.824049e+06,3.948220e+00
std,1.244405e+05,9.059209e+06,9.520989e-01
min,1.000000e+06,1.000000e+00,1.000000e+00
25%,1.107128e+06,1.705480e+05,3.000000e+00
50%,1.215328e+06,8.677937e+06,4.000000e+00
75%,1.322697e+06,1.718213e+07,5.000000e+00
max,1.430584e+06,3.642107e+07,5.000000e+00


# Разбиение с учётом хронологии

Рекомендательные системы на практике работают с учётом хронологии. Поэтому поток событий для тренировки и валидации полезно делить на то, что уже случилось, и что ещё случится. Это позволяет проводить валидацию на тех же пользователях, на которых тренировались, но на их событиях в будущем.

# === Знакомство: "холодный" старт

In [18]:
old_uid = '8f50136afeb65c55cec7b3d306c24b03'

interactions.query("user_id == @old_uid")

,user_id,book_id,started_at,read_at,is_read,rating,is_reviewed
10841363,8f50136afeb65c55cec7b3d306c24b03,253058,2012-12-29,2013-01-22,True,3,True
10841364,8f50136afeb65c55cec7b3d306c24b03,10572,2012-01-02,2013-01-27,True,5,False


In [21]:
events.query("item_id == 253058 and is_read == True and rating == 3 and is_reviewed == True")

,user_id,item_id,started_at,read_at,is_read,rating,is_reviewed
902241,1411824,253058,2014-05-06,2014-05-06,True,3,True
2370789,1141610,253058,2013-07-05,2014-04-17,True,3,True
3770520,1002928,253058,2016-04-16,2016-04-24,True,3,True
5906426,1227021,253058,2015-01-11,2015-01-17,True,3,True
6133010,1346812,253058,2015-06-12,2015-06-14,True,3,True
7043494,1329312,253058,2012-12-08,2012-12-28,True,3,True
8199826,1412609,253058,2011-09-17,2011-09-25,True,3,True
9292931,1301340,253058,2017-07-23,2017-07-30,True,3,True
10841363,1241243,253058,2012-12-29,2013-01-22,True,3,True
11013442,1267186,253058,2012-02-24,2012-03-06,True,3,True


In [5]:
# зададим точку разбиения
train_test_global_time_split_date = pd.to_datetime("2017-08-01").date()

train_test_global_time_split_idx = events["started_at"] < train_test_global_time_split_date
events_train = events[train_test_global_time_split_idx]
events_test = events[~train_test_global_time_split_idx]

# количество пользователей в train и test
users_train = events_train["user_id"].drop_duplicates()
users_test = events_test["user_id"].drop_duplicates()

# количество пользователей, которые есть и в train, и в test
common_users = set(users_train).intersection(set(users_test))

print(len(users_train), len(users_test), len(common_users))

428220 123223 120858


In [28]:
cold_users = set(users_test) - set(common_users)

print(len(cold_users)) 

2365


In [45]:
from sklearn.preprocessing import MinMaxScaler

top_pop_start_date = pd.to_datetime("2015-01-01").date()

item_popularity = events_train \
    .query("started_at >= @top_pop_start_date") \
    .groupby(["item_id"]).agg(users=("user_id", "nunique"), avg_rating=("rating", "mean")).reset_index()

# нормализация пользователей и среднего рейтинга, требуется для их приведения к одному масштабу
scaler = MinMaxScaler()
item_popularity[["users_norm", "avg_rating_norm"]] = scaler.fit_transform(
    item_popularity[["users", "avg_rating"]]
)

# вычисляем popularity_score, как скор популярности со штрафом за низкий рейтинг
item_popularity["popularity_score"] = (
    item_popularity["users_norm"] * item_popularity["avg_rating_norm"]
)



# сортируем по убыванию popularity_score
item_popularity = item_popularity.sort_values(['popularity_score'], ascending=False)

# выбираем первые 100 айтемов со средней оценкой avg_rating не меньше 4
top_k_pop_items = item_popularity.query("avg_rating >= 4").head(100)
top_k_pop_items

,item_id,users,avg_rating,users_norm,avg_rating_norm,popularity_score
32387,18007564,20207,4.321275,0.496596,0.830319,0.412333
32623,18143977,19462,4.290669,0.478287,0.822667,0.393471
2,3,15139,4.706057,0.372042,0.926514,0.344702
30695,16096824,16770,4.301014,0.412126,0.825253,0.340108
1916,15881,13043,4.632447,0.320529,0.908112,0.291076
...,...,...,...,...,...,...
24837,8490112,4792,4.080968,0.117747,0.770242,0.090694
33611,18966819,4361,4.374914,0.107154,0.843729,0.090409
378,3636,4667,4.098564,0.114675,0.774641,0.088832
32835,18293427,4674,4.092640,0.114847,0.773160,0.088795


In [46]:
# добавляем информацию о книгах
top_k_pop_items_with_info = top_k_pop_items.merge(
    items.set_index("item_id")[["author", "title", "genre_and_votes", "publication_year"]], on="item_id")

with pd.option_context('display.max_rows', 20):
    display(top_k_pop_items_with_info[["item_id", "author", "title", "publication_year", "users", "avg_rating", "popularity_score", "genre_and_votes"]])

,item_id,author,title,publication_year,users,avg_rating,popularity_score,genre_and_votes
0,18007564,Andy Weir,The Martian,2014,20207,4.321275,0.412333,"{'Science Fiction': 11966, 'Fiction': 8430}"
1,18143977,Anthony Doerr,All the Light We Cannot See,2014,19462,4.290669,0.393471,"{'Historical-Historical Fiction': 13679, 'Fict..."
2,3,"J.K. Rowling, Mary GrandPré",Harry Potter and the Sorcerer's Stone (Harry P...,1997,15139,4.706057,0.344702,"{'Fantasy': 59818, 'Fiction': 17918, 'Young Ad..."
3,16096824,Sarah J. Maas,A Court of Thorns and Roses (A Court of Thorns...,2015,16770,4.301014,0.340108,"{'Fantasy': 14326, 'Young Adult': 4662, 'Roman..."
4,15881,"J.K. Rowling, Mary GrandPré",Harry Potter and the Chamber of Secrets (Harry...,1999,13043,4.632447,0.291076,"{'Fantasy': 50130, 'Young Adult': 15202, 'Fict..."
...,...,...,...,...,...,...,...,...
95,8490112,Laini Taylor,Daughter of Smoke & Bone (Daughter of Smoke & ...,2011,4792,4.080968,0.090694,"{'Fantasy': 11681, 'Young Adult': 7110, 'Roman..."
96,18966819,Pierce Brown,"Golden Son (Red Rising, #2)",2015,4361,4.374914,0.090409,"{'Science Fiction': 2613, 'Fantasy': 1372, 'Sc..."
97,3636,Lois Lowry,"The Giver (The Giver, #1)",2006,4667,4.098564,0.088832,"{'Young Adult': 10993, 'Fiction': 9045, 'Class..."
98,18293427,Gabrielle Zevin,The Storied Life of A.J. Fikry,2014,4674,4.092640,0.088795,"{'Fiction': 3795, 'Contemporary': 1100, 'Writi..."


In [51]:
cold_users_events_with_recs = \
    events_test[events_test["user_id"].isin(cold_users)] \
    .merge(top_k_pop_items[["item_id", "avg_rating"]], on="item_id", how="left")

cold_user_items_no_avg_rating_idx = cold_users_events_with_recs["avg_rating"].isnull()
cold_user_recs = cold_users_events_with_recs[~cold_user_items_no_avg_rating_idx] \
    [["user_id", "item_id", "rating", "avg_rating"]]
cold_users_events_with_recs
    

,user_id,item_id,started_at,read_at,is_read,rating,is_reviewed,avg_rating
0,1361610,6900,2017-10-09,2017-10-13,True,4,False,NaN
1,1361610,12555,2017-09-21,2017-10-11,True,3,False,NaN
2,1361610,25899336,2017-09-12,2017-09-17,True,4,True,4.427261
3,1361610,21936809,2017-08-20,2017-08-24,True,4,True,NaN
4,1361610,6952,2017-09-18,2017-09-20,True,3,False,NaN
...,...,...,...,...,...,...,...,...
9667,1178502,252499,2017-09-30,2017-10-06,True,4,False,NaN
9668,1253160,51113,2017-09-25,2017-10-07,True,4,False,NaN
9669,1253160,16181775,2017-09-24,2017-09-25,True,3,False,NaN
9670,1253160,10210,2017-09-16,2017-09-24,True,5,False,NaN


In [59]:
cold_users_events_with_recs['avg_rating'].notna().sum()

1912

In [61]:
# посчитаем метрики рекомендаций
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

rmse = np.sqrt(
    mean_squared_error(
        cold_user_recs["rating"],
        cold_user_recs["avg_rating"]
    )
)

mae = mean_absolute_error(
    cold_user_recs["rating"],
    cold_user_recs["avg_rating"]
)

print(round(rmse, 2), round(mae, 2))

0.78 0.62


In [62]:
# посчитаем покрытие холодных пользователей рекомендациями

cold_users_hit_ratio = cold_users_events_with_recs.groupby("user_id").agg(hits=("avg_rating", lambda x: (~x.isnull()).mean()))

print(f"Доля пользователей без релевантных рекомендаций: {(cold_users_hit_ratio == 0).mean().iat[0]:.2f}")
print(f"Среднее покрытие пользователей: {cold_users_hit_ratio[cold_users_hit_ratio != 0].mean().iat[0]:.2f}")

Доля пользователей без релевантных рекомендаций: 0.59
Среднее покрытие пользователей: 0.44


# === Знакомство: первые персональные рекомендации

In [4]:
ui_data = events[['user_id', 'item_id', 'rating']]

n_users = ui_data['user_id'].nunique()
n_items = ui_data['item_id'].nunique()

all_cells = n_users * n_items
filled_cells = len(ui_data)

empty_cells = all_cells - filled_cells

sparsity = empty_cells / all_cells

print(sparsity)

0.9993451160571009


In [64]:
from surprise import Dataset, Reader
from surprise import SVD

# используем Reader из библиотеки surprise для преобразования событий (events)
# в формат, необходимый surprise
reader = Reader(rating_scale=(1, 5))
surprise_train_set = Dataset.load_from_df(events_train[['user_id', 'item_id', 'rating']], reader)
surprise_train_set = surprise_train_set.build_full_trainset()

# инициализируем модель
svd_model = SVD(n_factors=100, random_state=0)

# обучаем модель
svd_model.fit(surprise_train_set)

In [65]:
surprise_test_set = list(events_test[['user_id', 'item_id', 'rating']].itertuples(index=False))

# получаем рекомендации для тестовой выборки
svd_predictions = svd_model.test(surprise_test_set)

In [66]:
from surprise import accuracy

rmse = accuracy.rmse(svd_predictions)
mae = accuracy.mae(svd_predictions)
                     
print(rmse, mae)

RMSE: 0.8289
MAE:  0.6474
0.8288711689059135 0.647437483750257


In [67]:
from surprise import NormalPredictor

# инициализируем состояние генератора, это необходимо для получения
# одной и той же последовательности случайных чисел, только в учебных целях
np.random.seed(0)

random_model = NormalPredictor()

random_model.fit(surprise_train_set)
random_predictions = random_model.test(surprise_test_set)

In [68]:
rmse_rand = accuracy.rmse(random_predictions)
mae_rand = accuracy.mae(random_predictions)
                     
print(rmse_rand, mae_rand)

RMSE: 1.2628
MAE:  1.0018
1.2628030301013033 1.0017726877569562


In [69]:
events

,user_id,item_id,started_at,read_at,is_read,rating,is_reviewed
0,1229132,22034,2015-07-12,2015-07-17,True,5,False
1,1229132,22318578,2015-06-07,2015-08-09,True,5,True
2,1229132,22551730,2015-06-24,2015-07-11,True,4,True
3,1229132,22816087,2015-09-27,2015-11-04,True,5,True
5,1229132,17910054,2015-03-04,2015-07-28,True,3,False
...,...,...,...,...,...,...,...
12914452,1364473,5297,2017-02-07,2017-02-26,True,5,False
12914453,1364473,4900,2016-12-22,2016-12-29,True,2,False
12914454,1364473,14836,2016-11-29,2017-01-15,True,3,False
12914456,1297020,10210,2012-06-05,2013-01-17,True,5,False


In [76]:
def get_recommendations_svd(user_id, all_items, events, model, include_seen=True, n=5):

    """ возвращает n рекомендаций для user_id """
    
    # получим список идентификаторов всех книг
    all_items = set(events['item_id'].unique())
        
    # учитываем флаг, стоит ли уже прочитанные книги включать в рекомендации
    if include_seen:
        items_to_predict = list(all_items)
    else:
        # получим список книг, которые пользователь уже прочитал ("видел")
        seen_items = set(
            events[
                (events['user_id'] == user_id) &
                (events['is_read'] == True)
            ]['item_id'].unique()
        )
        
        # книги, которые пользователь ещё не читал
        # только их и будем включать в рекомендации
        items_to_predict = list(all_items - seen_items)
    
    # получаем скоры для списка книг, т. е. рекомендации
    predictions = [model.predict(user_id, item_id) for item_id in items_to_predict]
    
    # сортируем рекомендации по убыванию скора и берём только n первых
    recommendations = sorted(predictions, key=lambda x: x.est, reverse=True)[:n]
    
    return pd.DataFrame([(pred.iid, pred.est) for pred in recommendations], columns=["item_id", "score"])

print(get_recommendations_svd(1296647, items, events_test, svd_model))

    item_id     score
0   7864312  4.981188
1  25793186  4.912001
2  12174312  4.898052
3     13208  4.894869
4  33353628  4.891661


In [77]:
# выберем произвольного пользователя из тренировочной выборки ("прошлого")
user_id = events_train['user_id'].sample().iat[0]

print(f"user_id: {user_id}")

print("История (последние события, recent)")
user_history = (
    events_train
    .query("user_id == @user_id")
    .merge(items.set_index("item_id")[["author", "title", "genre_and_votes"]], on="item_id")
)
user_history_to_print = user_history[["author", "title", "started_at", "read_at", "rating", "genre_and_votes"]].tail(10)
display(user_history_to_print)

print("Рекомендации")
user_recommendations = get_recommendations_svd(user_id, items, events_train, svd_model)
user_recommendations = user_recommendations.merge(items[["item_id", "author", "title", "genre_and_votes"]], on="item_id")
display(user_recommendations)

user_id: 1169023
История (последние события, recent)


,author,title,started_at,read_at,rating,genre_and_votes
68,Veronica Roth,"Divergent (Divergent, #1)",2014-06-02,2014-06-04,4,"{'Young Adult': 20260, 'Science Fiction-Dystop..."
69,"Gillian Flynn, В. Русанов",Gone Girl,2014-05-27,2014-05-29,5,"{'Fiction': 11773, 'Mystery': 9965, 'Thriller'..."
70,Kathy Reichs,"Death du Jour (Temperance Brennan, #2)",2014-05-24,2014-05-27,4,"{'Mystery': 1206, 'Mystery-Crime': 579, 'Ficti..."
71,Chelsea Cain,"Heartsick (Archie Sheridan & Gretchen Lowell, #1)",2014-05-22,2014-05-22,5,"{'Mystery': 832, 'Thriller': 653, 'Fiction': 4..."
72,"Jussi Adler-Olsen, Lisa Hartford","The Keeper of Lost Causes (Department Q, #1)",2014-05-30,2014-06-02,3,"{'Mystery': 1225, 'Mystery-Crime': 627, 'Ficti..."
73,Gillian Flynn,Dark Places,2014-05-17,2014-05-22,4,"{'Mystery': 4534, 'Fiction': 4055, 'Thriller':..."
74,Audrey Niffenegger,Her Fearful Symmetry,2014-05-05,2014-05-08,2,"{'Fiction': 1984, 'Fantasy': 674, 'Fantasy-Par..."
75,Kathy Reichs,"Déjà Dead (Temperance Brennan, #1)",2014-05-13,2014-05-17,4,"{'Mystery': 2141, 'Fiction': 904, 'Mystery-Cri..."
76,Carolyn Parkhurst,The Dogs of Babel,2014-05-09,2014-05-10,5,"{'Fiction': 522, 'Mystery': 102, 'Animals': 77..."
77,George R.R. Martin,"A Dance with Dragons (A Song of Ice and Fire, #5)",2014-05-04,2014-05-04,5,"{'Fantasy': 22247, 'Fiction': 4512, 'Fantasy-E..."


Рекомендации


,item_id,score,author,title,genre_and_votes
0,2199,5,Doris Kearns Goodwin,Team of Rivals: The Political Genius of Abraha...,"{'History': 4174, 'Nonfiction': 2127, 'Biograp..."
1,16255632,5,"David Gaider, Ben Gelinas, Mike Laidlaw, Dave ...",Dragon Age: The World of Thedas Volume 1,"{'Fantasy': 134, 'Games-Video Games': 28, 'Art..."
2,2363958,5,João Guimarães Rosa,Grande Sertão: Veredas,"{'Fiction': 85, 'Classics': 69, 'Cultural-Braz..."
3,22552026,5,Jason Reynolds,Long Way Down,"{'Young Adult': 1871, 'Poetry': 1737, 'Contemp..."
4,29237211,5,"Brian K. Vaughan, Fiona Staples","Saga, Vol. 7 (Saga, #7)","{'Sequential Art-Graphic Novels': 2539, 'Seque..."


In [84]:
new_user_id = events["user_id"].max() + 1

new_user_items = [7144, 3, 15881]

# создаём новые события
new_user_events = pd.DataFrame({
    "user_id": [new_user_id] * len(new_user_items),
    "item_id": new_user_items,
    "rating": [5] * len(new_user_items),   # можно поставить свои оценки
    "is_read": [True] * len(new_user_items)
})

# добавляем в events
events_extended = pd.concat(
    [events, new_user_events],
    ignore_index=True
)

In [87]:
recs = get_recommendations_svd(
    user_id=new_user_id,
    all_items=events_extended["item_id"].unique(),
    events=events_extended,
    model=svd_model,
    include_seen=False,
    n=5
)
recs = recs.merge(items[["item_id", "author", "title", "genre_and_votes"]], on="item_id")
display(recs)

,item_id,score,author,title,genre_and_votes
0,24812,5.000000,Bill Watterson,The Complete Calvin and Hobbes,"{'Sequential Art-Comics': 867, 'Humor': 378, '..."
1,11221285,4.914296,Brandon Sanderson,"The Way of Kings, Part 2 (The Stormlight Archi...","{'Fantasy': 641, 'Fiction': 46, 'Fantasy-Epic ..."
2,22037424,4.908423,"J.K. Rowling, Jonny Duddle, Tomislav Tomić",Harry Potter and the Prisoner of Azkaban (Harr...,"{'Fantasy': 49994, 'Young Adult': 15433, 'Fict..."
3,33353628,4.872179,Pénélope Bagieu,"Culottées #2 (Culottées, #2)","{'Sequential Art-Bande DessinÃ©e': 108, 'Femin..."
4,54741,4.872000,Quino,Toda Mafalda,"{'Sequential Art-Comics': 157, 'Humor': 47, 'S..."


# === Базовые подходы: коллаборативная фильтрация

In [6]:
import scipy
import sklearn.preprocessing

# перекодируем идентификаторы пользователей: 
# из имеющихся в последовательность 0, 1, 2, ...
user_encoder = sklearn.preprocessing.LabelEncoder()
user_encoder.fit(events["user_id"])
events_train["user_id_enc"] = user_encoder.transform(events_train["user_id"])
events_test["user_id_enc"] = user_encoder.transform(events_test["user_id"])

# перекодируем идентификаторы объектов: 
# из имеющихся в последовательность 0, 1, 2, ...
item_encoder = sklearn.preprocessing.LabelEncoder()
item_encoder.fit(items["item_id"])
items["item_id_enc"] = item_encoder.transform(items["item_id"])
events_train["item_id_enc"] = item_encoder.transform(events_train["item_id"])
events_test["item_id_enc"] = item_encoder.transform(events_test["item_id"])

/tmp/ipykernel_2398/1044897688.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  events_train["user_id_enc"] = user_encoder.transform(events_train["user_id"])
/tmp/ipykernel_2398/1044897688.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  events_test["user_id_enc"] = user_encoder.transform(events_test["user_id"])
/tmp/ipykernel_2398/1044897688.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See t

In [20]:
events_train['item_id_enc'].max()

43304

In [21]:
items['title'].nunique() * events_train['user_id'].nunique() / 1024**3

17.065120674669743

In [7]:
# создаём sparse-матрицу формата CSR 
user_item_matrix_train = scipy.sparse.csr_matrix((
    events_train["rating"],
    (events_train['user_id_enc'], events_train['item_id_enc'])),
    dtype=np.int8)

In [23]:
import sys

sum([sys.getsizeof(i) for i in user_item_matrix_train.data])/1024**3

0.26370687410235405

In [8]:
from implicit.als import AlternatingLeastSquares

als_model = AlternatingLeastSquares(factors=50, iterations=50, regularization=0.05, random_state=0)
als_model.fit(user_item_matrix_train)

/home/mle-user/mle_projects/mle-recsys-start/env_recsys_start/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/mle-user/mle_projects/mle-recsys-start/env_recsys_start/lib/python3.10/site-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 4 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████| 50/50 [02:51<00:00,  3.44s/it]


In [9]:
def get_recommendations_als(user_item_matrix, model, user_id, user_encoder, item_encoder, include_seen=True, n=5):
    """
    Возвращает отранжированные рекомендации для заданного пользователя
    """
    user_id_enc = user_encoder.transform([user_id])[0]
    recommendations = model.recommend(
         user_id_enc, 
         user_item_matrix[user_id_enc], 
         filter_already_liked_items=not include_seen,
         N=n)
    recommendations = pd.DataFrame({"item_id_enc": recommendations[0], "score": recommendations[1]})
    recommendations["item_id"] = item_encoder.inverse_transform(recommendations["item_id_enc"])
    
    return recommendations

In [28]:
# выберем произвольного пользователя из тренировочной выборки ("прошлого")
user_id = events_train['user_id'].sample().iat[0]

print(f"user_id: {user_id}")

print("История (последние события, recent)")
user_history = (
    events_train
    .query("user_id == @user_id")
    .merge(items.set_index("item_id")[["author", "title", "genre_and_votes"]], on="item_id")
)
user_history_to_print = user_history[["author", "title", "started_at", "read_at", "rating", "genre_and_votes"]].tail(10)
display(user_history_to_print)

print("Рекомендации")
user_recommendations = get_recommendations_als(user_item_matrix_train, als_model, user_id, user_encoder, item_encoder)
user_recommendations = user_recommendations.merge(items[["item_id", "author", "title", "genre_and_votes"]], on="item_id")
display(user_recommendations)

user_id: 1021681
История (последние события, recent)


,author,title,started_at,read_at,rating,genre_and_votes
95,Veronica Rossi,"Through the Ever Night (Under the Never Sky, #2)",2013-03-25,2013-03-29,4,"{'Young Adult': 1407, 'Science Fiction-Dystopi..."
96,Lauren Oliver,"Requiem (Delirium, #3)",2013-03-31,2013-04-05,4,"{'Young Adult': 2521, 'Science Fiction-Dystopi..."
97,Kelley Armstrong,"No Humans Involved (Women of the Otherworld, #7)",2013-03-05,2013-03-11,4,"{'Fantasy-Urban Fantasy': 764, 'Fantasy-Parano..."
98,Pittacus Lore,"The Power of Six (Lorien Legacies, #2)",2013-02-21,2013-03-05,2,"{'Young Adult': 1543, 'Fantasy': 1163, 'Scienc..."
99,George R.R. Martin,"A Storm of Swords (A Song of Ice and Fire, #3)",2013-01-24,2013-02-13,4,"{'Fantasy': 26263, 'Fiction': 5489, 'Fantasy-E..."
100,James Dashner,"The Kill Order (Maze Runner, #0.5)",2013-01-17,2013-01-20,4,"{'Young Adult': 1715, 'Science Fiction-Dystopi..."
101,Kelley Armstrong,"Haunted (Women of the Otherworld, #5)",2013-01-12,2013-01-17,3,"{'Fantasy-Urban Fantasy': 774, 'Fantasy': 543,..."
102,Julie Kagawa,"The Immortal Rules (Blood of Eden, #1)",2012-12-30,2013-01-07,2,"{'Paranormal-Vampires': 1680, 'Young Adult': 1..."
103,Michael Grant,"Plague (Gone, #4)",2011-04-13,2013-05-02,4,"{'Young Adult': 741, 'Science Fiction-Dystopia..."
104,Cassandra Clare,"City of Fallen Angels (The Mortal Instruments,...",2011-04-09,2011-04-13,1,"{'Fantasy': 8553, 'Young Adult': 6438, 'Fantas..."


Рекомендации


,item_id_enc,score,item_id,author,title,genre_and_votes
0,38878,1.076010,22557272,Paula Hawkins,The Girl on the Train,"{'Fiction': 9793, 'Mystery': 9190, 'Thriller':..."
1,1641,0.925027,13496,George R.R. Martin,"A Game of Thrones (A Song of Ice and Fire, #1)","{'Fantasy': 44086, 'Fiction': 10111, 'Fantasy-..."
2,23331,0.869362,6186357,James Dashner,"The Maze Runner (Maze Runner, #1)","{'Young Adult': 10127, 'Science Fiction-Dystop..."
3,25839,0.865532,7631105,James Dashner,"The Scorch Trials (Maze Runner, #2)","{'Young Adult': 5434, 'Science Fiction-Dystopi..."
4,1232,0.859056,10572,George R.R. Martin,"A Clash of Kings (A Song of Ice and Fire, #2)","{'Fantasy': 31452, 'Fiction': 6543, 'Fantasy-E..."


In [11]:
# получаем список всех возможных user_id (перекодированных)
user_ids_encoded = range(len(user_encoder.classes_))

# получаем рекомендации для всех пользователей
als_recommendations = als_model.recommend(
    user_ids_encoded, 
    user_item_matrix_train[user_ids_encoded], 
    filter_already_liked_items=False, N=100)

# преобразуем полученные рекомендации в табличный формат
item_ids_enc = als_recommendations[0]
als_scores = als_recommendations[1]

als_recommendations = pd.DataFrame({
    "user_id_enc": user_ids_encoded,
    "item_id_enc": item_ids_enc.tolist(), 
    "score": als_scores.tolist()})
als_recommendations = als_recommendations.explode(["item_id_enc", "score"], ignore_index=True)

# приводим типы данных
als_recommendations["item_id_enc"] = als_recommendations["item_id_enc"].astype("int")
als_recommendations["score"] = als_recommendations["score"].astype("float")

# получаем изначальные идентификаторы
als_recommendations["user_id"] = user_encoder.inverse_transform(als_recommendations["user_id_enc"])
als_recommendations["item_id"] = item_encoder.inverse_transform(als_recommendations["item_id_enc"])
als_recommendations = als_recommendations.drop(columns=["user_id_enc", "item_id_enc"])

In [12]:
als_recommendations = als_recommendations[["user_id", "item_id", "score"]]
als_recommendations.to_parquet("als_recommendations.parquet")

In [ ]:
als_recommendations = (
    als_recommendations     
    .merge(events_test[["user_id", "item_id", "rating"]]
               .rename(columns={"rating": "rating_test"}), 
           on=["user_id", "item_id"], how="left")
)

In [10]:
import sklearn.metrics

def compute_ndcg(rating: pd.Series, score: pd.Series, k):

    """ подсчёт ndcg
    rating: истинные оценки
    score: оценки модели
    k: количество айтемов (по убыванию score) для оценки, остальные - отбрасываются
    """
    
    # если кол-во объектов меньше 2, то NDCG - не определена
    if len(rating) < 2:
        return np.nan

    ndcg = sklearn.metrics.ndcg_score(np.asarray([rating.to_numpy()]), np.asarray([score.to_numpy()]), k=k)

    return ndcg

In [33]:
rating_test_idx = ~als_recommendations["rating_test"].isnull()
ndcg_at_5_scores = als_recommendations[rating_test_idx].groupby("user_id").apply(lambda x: compute_ndcg(x["rating_test"], x["score"], k=5))
print(ndcg_at_5_scores.mean())

0.975946709792109


/tmp/ipykernel_2473/1713422991.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ndcg_at_5_scores = als_recommendations[rating_test_idx].groupby("user_id").apply(lambda x: compute_ndcg(x["rating_test"], x["score"], k=5))


# === Базовые подходы: контентные рекомендации

In [7]:
import pandas as pd
items = pd.read_parquet('items.par')
events = pd.read_parquet('events.par')

In [8]:
items["genre_and_votes"] = items["genre_and_votes"].apply(eval)

In [13]:
import scipy
import sklearn.preprocessing

# перекодируем идентификаторы пользователей: 
# из имеющихся в последовательность 0, 1, 2, ...
user_encoder = sklearn.preprocessing.LabelEncoder()
user_encoder.fit(events["user_id"])
events_train["user_id_enc"] = user_encoder.transform(events_train["user_id"])
events_test["user_id_enc"] = user_encoder.transform(events_test["user_id"])

# перекодируем идентификаторы объектов: 
# из имеющихся в последовательность 0, 1, 2, ...
item_encoder = sklearn.preprocessing.LabelEncoder()
item_encoder.fit(items["item_id"])
items["item_id_enc"] = item_encoder.transform(items["item_id"])
events_train["item_id_enc"] = item_encoder.transform(events_train["item_id"])
events_test["item_id_enc"] = item_encoder.transform(events_test["item_id"])

/tmp/ipykernel_3375/1044897688.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  events_train["user_id_enc"] = user_encoder.transform(events_train["user_id"])
/tmp/ipykernel_3375/1044897688.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  events_test["user_id_enc"] = user_encoder.transform(events_test["user_id"])
/tmp/ipykernel_3375/1044897688.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See t

In [14]:
items.head(1)

,item_id,author,title,description,genre_and_votes,num_pages,average_rating,ratings_count,text_reviews_count,publisher,publication_year,country_code,language_code,format,is_ebook,isbn,isbn13,genre_and_votes_dict,genre_and_votes_str,item_id_enc
3,6066819,Jennifer Weiner,Best Friends Forever,Addie Downs and Valerie Adler were eight when ...,"{'Womens Fiction-Chick Lit': 739, 'Fiction': 442}",368,3.49,51184,3282,Atria Books,2009,US,eng,Hardcover,False,0743294297,9780743294294,"{'Academic': None, 'Academic-Academia': None, ...","Womens Fiction-Chick Lit 739, Fiction 442",23133


In [15]:
def get_genres(items):

    """ 
    извлекает список жанров по всем книгам, 
    подсчитывает долю голосов по каждому их них
    """
    
    genres_counter = {}
    
    for k, v, in items.iterrows():
        genre_and_votes = v['genre_and_votes']
        if genre_and_votes is None or not isinstance(genre_and_votes, dict):
            continue
        for genre, votes in genre_and_votes.items():
            # увеличиваем счётчик жанров
            try:
                genres_counter[genre] += votes
            except KeyError:
                genres_counter[genre] = 0

    genres = pd.Series(genres_counter, name="votes")
    genres = genres.to_frame()
    genres = genres.reset_index().rename(columns={"index": "name"})
    genres.index.name = "genre_id"
    
    return genres
   
genres = get_genres(items)
display(genres)

,name,votes
genre_id,,
0,Womens Fiction-Chick Lit,254558
1,Fiction,6406256
2,Politics,103296
3,Humor,304302
4,Christian,105273
...,...,...
810,German History-Nazi Party,0
811,Favorites,0
812,History-Latin American History,0


In [16]:
genres["score"] = genres["votes"] / genres["votes"].sum()
genres.sort_values(by="score", ascending=False).head(10)

,name,votes,score
genre_id,,,
25,Fantasy,6850060,0.149651
1,Fiction,6406256,0.139955
38,Classics,3414934,0.074605
18,Young Adult,3296951,0.072027
34,Romance,2422614,0.052926
5,Nonfiction,1737406,0.037957
16,Historical-Historical Fiction,1531205,0.033452
20,Mystery,1371196,0.029956
24,Science Fiction,1218917,0.026629


In [17]:
def get_item2genre_matrix(genres, items):

    genre_names_to_id = genres.reset_index().set_index("name")["genre_id"].to_dict()
    
    # list to build CSR matrix
    genres_csr_data = []
    genres_csr_row_idx = []
    genres_csr_col_idx = []
    
    for item_idx, (k, v) in enumerate(items.iterrows()):
        if v["genre_and_votes"] is None:
            continue
        for genre_name, votes in v["genre_and_votes"].items():
            genre_idx = genre_names_to_id[genre_name]
            genres_csr_data.append(int(votes))
            genres_csr_row_idx.append(item_idx)
            genres_csr_col_idx.append(genre_idx)

    genres_csr = scipy.sparse.csr_matrix((genres_csr_data, (genres_csr_row_idx, genres_csr_col_idx)), shape=(len(items), len(genres)))
    # нормализуем, чтобы сумма оценок принадлежности к жанру была равна 1
    genres_csr = sklearn.preprocessing.normalize(genres_csr, norm='l1', axis=1)
    
    return genres_csr

items = items.sort_values(by="item_id_enc")
all_items_genres_csr = get_item2genre_matrix(genres, items)

In [46]:
user_id = 1000010
user_events = events_train.query("user_id == @user_id")[["item_id", "rating"]]
user_items = items[items["item_id"].isin(user_events["item_id"])]

user_items_genres_csr = get_item2genre_matrix(get_genres(items), user_items)
user_items_genres_csr

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 149 stored elements and shape (22, 815)>

In [47]:
# вычислим склонность пользователя к жанрам как среднее взвешенное значение популяции на его оценки книг.

# преобразуем пользовательские оценки из списка в вектор-столбец
user_ratings = user_events["rating"].to_numpy() / 5
user_ratings = np.expand_dims(user_ratings, axis=1)

user_items_genres_weighted = user_items_genres_csr.multiply(user_ratings)

user_genres_scores = np.asarray(user_items_genres_weighted.mean(axis=0))

# выведем список жанров, которые предпочитает пользователь

user_genres = genres.copy()
user_genres["score"] = np.ravel(user_genres_scores)
user_genres = user_genres[user_genres["score"] > 0].sort_values(by=["score"], ascending=False)

user_genres.head(5)

,name,votes,score
genre_id,,,
2,Politics,103296,0.185241
9,Autobiography-Memoir,266848,0.103879
0,Womens Fiction-Chick Lit,254558,0.072447
17,Childrens-Middle Grade,179782,0.050865
14,Religion-Theology,24218,0.040920


In [51]:
from sklearn.metrics.pairwise import cosine_similarity

# вычисляем сходство между вектором пользователя и векторами по книгам
similarity_scores = cosine_similarity(all_items_genres_csr, user_genres_scores)

# преобразуем в одномерный массив
similarity_scores = similarity_scores.flatten()

# получаем индексы top-k
k = 5
top_k_indices = np.argsort(similarity_scores)[::-1][:k]

selected_items = items[items["item_id_enc"].isin(top_k_indices)]

with pd.option_context("max_colwidth", 100):
   display(selected_items[["author", "title", "genre_and_votes"]])

,author,title,genre_and_votes
804309,Andrew Hosken,Nothing Like a Dame: The Scandals of Shirley Porter,{'Politics': 1}
674166,بلال فضل,قلمين,{'Politics': 8}
2114715,جمال عبد الناصر,فلسفة الثورة,"{'Politics': 14, 'History': 3}"
1882486,مصطفى محمود,الغد المشتعل,{'Politics': 6}
1793067,"Claus Kleber, Cleo Paskal, Thomas Pfeiffer",Spielball Erde: Machtkämpfe im Klimawandel,{'Politics': 1}


# === Базовые подходы: валидация

In [42]:
def process_events_recs_for_binary_metrics(events_train, events_test, recs, top_k=None):

    """
    размечает пары <user_id, item_id> для общего множества пользователей признаками
    - gt (ground truth)
    - pr (prediction)
    top_k: расчёт ведётся только для top k-рекомендаций
    """

    events_test["gt"] = True
    common_users = set(events_test["user_id"]) & set(recs["user_id"])

    print(f"Common users: {len(common_users)}")
    
    events_for_common_users = events_test[events_test["user_id"].isin(common_users)].copy()
    recs_for_common_users = recs[recs["user_id"].isin(common_users)].copy()

    recs_for_common_users = recs_for_common_users.sort_values(["user_id", "score"], ascending=[True, False])

    # оставляет только те item_id, которые были в events_train, 
    # т. к. модель не имела никакой возможности давать рекомендации для новых айтемов
    events_for_common_users = events_for_common_users[events_for_common_users["item_id"].isin(events_train["item_id"].unique())]

    if top_k is not None:
        recs_for_common_users = recs_for_common_users.groupby("user_id").head(top_k)
    
    events_recs_common = events_for_common_users[["user_id", "item_id", "gt"]].merge(
        recs_for_common_users[["user_id", "item_id", "score"]], 
        on=["user_id", "item_id"], how="outer")    

    events_recs_common["gt"] = events_recs_common["gt"].fillna(False)
    events_recs_common["pr"] = ~events_recs_common["score"].isnull()
    
    events_recs_common["tp"] = events_recs_common["gt"] & events_recs_common["pr"]
    events_recs_common["fp"] = ~events_recs_common["gt"] & events_recs_common["pr"]
    events_recs_common["fn"] = events_recs_common["gt"] & ~events_recs_common["pr"]

    return events_recs_common

events_recs_for_binary_metrics = process_events_recs_for_binary_metrics(
  events_train,
    events_test, 
    als_recommendations, 
    top_k=5)

/tmp/ipykernel_3375/3370395532.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  events_test["gt"] = True


Common users: 123223


/tmp/ipykernel_3375/3370395532.py:31: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  events_recs_common["gt"] = events_recs_common["gt"].fillna(False)


In [44]:
def compute_cls_metrics(events_recs_for_binary_metric):
    
    groupper = events_recs_for_binary_metric.groupby("user_id")

    # precision = tp / (tp + fp)
    precision = groupper["tp"].sum()/(groupper["tp"].sum()+groupper["fp"].sum())
    precision = precision.fillna(0).mean()
    
    # recall = tp / (tp + fn)
    recall = groupper['tp'].sum()/(groupper['tp'].sum() + groupper['fn'].sum())
    recall = recall.fillna(0).mean()
    
    return precision, recall

precision_5, recall_5 = compute_cls_metrics(events_recs_for_binary_metrics)
print(precision_5, recall_5)

0.007617084472866265 0.014090629647930208


In [18]:
events_recs_for_binary_metrics_10 = process_events_recs_for_binary_metrics(
  events_train,
    events_test, 
    als_recommendations, 
    top_k=10)

precision_10, recall_10 = compute_cls_metrics(events_recs_for_binary_metrics_10)
print(precision_10, recall_10)

/tmp/ipykernel_1921/3370395532.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  events_test["gt"] = True


Common users: 123223
0.008732947582837622 0.03130238527136974


/tmp/ipykernel_1921/3370395532.py:31: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  events_recs_common["gt"] = events_recs_common["gt"].fillna(False)


# === Двухстадийный подход: метрики

In [18]:
import pandas as pd
als_recommendations = pd.read_parquet('als_recommendations.parquet')
display(als_recommendations)


,user_id,item_id,score
0,1000000,3,0.990941
1,1000000,15881,0.896617
2,1000000,5,0.864404
3,1000000,6,0.822254
4,1000000,2,0.774095
...,...,...,...
43058495,1430584,13206900,0.096082
43058496,1430584,5060378,0.096065
43058497,1430584,16071764,0.094949
43058498,1430584,9969571,0.094927


In [2]:
items = pd.read_parquet('items.par')
events = pd.read_parquet('events.par')

In [6]:
# расчёт покрытия по объектам
cov_items = len(set(als_recommendations['item_id'].tolist())) / len(items['item_id'].tolist())
print(f"{cov_items:.2f}")

0.09


In [19]:
import scipy
import sklearn.preprocessing

# зададим точку разбиения
train_test_global_time_split_date = pd.to_datetime("2017-08-01").date()

train_test_global_time_split_idx = events["started_at"] < train_test_global_time_split_date
events_train = events[train_test_global_time_split_idx]
events_test = events[~train_test_global_time_split_idx]

# количество пользователей в train и test
users_train = events_train["user_id"].drop_duplicates()
users_test = events_test["user_id"].drop_duplicates()

# перекодируем идентификаторы пользователей: 
# из имеющихся в последовательность 0, 1, 2, ...
user_encoder = sklearn.preprocessing.LabelEncoder()
user_encoder.fit(events["user_id"])
events_train["user_id_enc"] = user_encoder.transform(events_train["user_id"])
events_test["user_id_enc"] = user_encoder.transform(events_test["user_id"])

# перекодируем идентификаторы объектов: 
# из имеющихся в последовательность 0, 1, 2, ...
item_encoder = sklearn.preprocessing.LabelEncoder()
item_encoder.fit(items["item_id"])
items["item_id_enc"] = item_encoder.transform(items["item_id"])
events_train["item_id_enc"] = item_encoder.transform(events_train["item_id"])
events_test["item_id_enc"] = item_encoder.transform(events_test["item_id"])

/tmp/ipykernel_3375/1903076585.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  events_train["user_id_enc"] = user_encoder.transform(events_train["user_id"])
/tmp/ipykernel_3375/1903076585.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  events_test["user_id_enc"] = user_encoder.transform(events_test["user_id"])
/tmp/ipykernel_3375/1903076585.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See

In [4]:
# разметим каждую рекомендацию признаком read
events_train["read"] = True
als_recommendations = als_recommendations.merge(events_train[["user_id", "item_id", "read"]], on=["user_id", "item_id"], how="left")
als_recommendations["read"] = als_recommendations["read"].fillna(False).astype("bool")

# проставим ранги
als_recommendations = als_recommendations.sort_values(["user_id", "score"], ascending=[True, False])
als_recommendations["rank"] = als_recommendations.groupby("user_id").cumcount() + 1

# посчитаем novelty по пользователям
novelty_5 = (1-als_recommendations.query("rank <= 5").groupby("user_id")["read"].mean())

# посчитаем средний novelty
mean_novelty_5 = novelty_5.mean()
print(mean_novelty_5)

/tmp/ipykernel_2318/3198206351.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  events_train["read"] = True
/tmp/ipykernel_2318/3198206351.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  als_recommendations["read"] = als_recommendations["read"].fillna(False).astype("bool")


0.607333279143491


# === Двухстадийный подход: модель

In [20]:
display(events_test)

,user_id,item_id,started_at,read_at,is_read,rating,is_reviewed,user_id_enc,item_id_enc
84,1196635,18467802,2017-09-01,2017-09-22,True,1,False,196635,36588
257,1188739,10799,2017-08-06,2017-10-14,True,3,False,188739,1262
270,1001879,13206828,2017-09-24,2017-09-28,True,5,True,1879,31274
271,1001879,13206900,2017-09-28,2017-10-07,True,4,True,1879,31275
272,1001879,13206760,2017-09-21,2017-09-24,True,5,True,1879,31273
...,...,...,...,...,...,...,...,...,...
12913921,1145655,25781157,2017-08-04,2017-08-14,True,3,False,145655,40870
12914084,1166047,92057,2017-08-20,2017-09-17,True,5,False,166047,6577
12914085,1166047,168668,2017-08-03,2017-08-19,True,2,False,166047,9075
12914150,1155073,25300956,2017-08-12,2017-10-14,True,5,False,155073,40537


In [21]:
# задаём точку разбиения
split_date_for_labels = pd.to_datetime("2017-09-15").date()

split_date_for_labels_idx = events_test["started_at"] < split_date_for_labels
events_labels = events_test[split_date_for_labels_idx].copy()
events_test_2 = events_test[~split_date_for_labels_idx].copy()
print(len(set(events_labels['user_id'].tolist())))
display(events_labels)

99849


,user_id,item_id,started_at,read_at,is_read,rating,is_reviewed,user_id_enc,item_id_enc
84,1196635,18467802,2017-09-01,2017-09-22,True,1,False,196635,36588
257,1188739,10799,2017-08-06,2017-10-14,True,3,False,188739,1262
273,1001879,18658071,2017-09-11,2017-09-15,True,2,True,1879,36848
274,1001879,25785357,2017-08-23,2017-08-26,True,4,False,1879,40875
275,1001879,22557272,2017-08-15,2017-08-21,True,3,False,1879,38878
...,...,...,...,...,...,...,...,...,...
12913921,1145655,25781157,2017-08-04,2017-08-14,True,3,False,145655,40870
12914084,1166047,92057,2017-08-20,2017-09-17,True,5,False,166047,6577
12914085,1166047,168668,2017-08-03,2017-08-19,True,2,False,166047,9075
12914150,1155073,25300956,2017-08-12,2017-10-14,True,5,False,155073,40537


In [22]:
# загружаем рекомендации от двух базовых генераторов
als_recommendations = pd.read_parquet("candidates/training/als_recommendations.parquet")
content_recommendations = pd.read_parquet("candidates/training/content_recommendations.parquet")

candidates = pd.merge(
    als_recommendations[["user_id", "item_id", "score"]].rename(columns={"score": "als_score"}),
    content_recommendations[["user_id", "item_id", "score"]].rename(columns={"score": "cnt_score"}),
    on=['user_id', 'item_id'],
    how="outer")

candidates.shape

(82993094, 4)

In [23]:
# добавляем таргет к кандидатам со значением:
# — 1 для тех item_id, которые пользователь прочитал
# — 0, для всех остальных 

events_labels["target"] = 1
candidates_merged = candidates.merge(events_labels[["user_id", "item_id", "target"]], 
                              on=['user_id', 'item_id'], how="left")
candidates_merged["target"] = candidates_merged["target"].fillna(0).astype("int")
display(candidates_merged.describe())
# в кандидатах оставляем только тех пользователей, у которых есть хотя бы один положительный таргет
candidates_to_sample = candidates_merged.groupby("user_id").filter(lambda x: x["target"].sum() > 0)

# для каждого пользователя оставляем только 4 негативных примера
negatives_per_user = 4
candidates_for_train = pd.concat([
    candidates_to_sample.query("target == 1"),
    candidates_to_sample.query("target == 0") \
        .groupby("user_id") \
        .apply(lambda x: x.sample(min(len(x), negatives_per_user), random_state=0))
    ])

candidates_for_train.shape

,user_id,item_id,als_score,cnt_score,target
count,8.299309e+07,8.299309e+07,4.305850e+07,4.282200e+07,8.299309e+07
mean,1.215275e+06,8.599908e+06,1.999375e-01,8.792080e-01,7.047815e-04
std,1.243038e+05,9.199594e+06,2.163485e-01,8.048963e-02,2.653837e-02
min,1.000000e+06,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.107625e+06,9.975700e+04,5.178498e-02,8.342599e-01,0.000000e+00
50%,1.215272e+06,6.411961e+06,1.166837e-01,8.955294e-01,0.000000e+00
75%,1.322927e+06,1.584244e+07,2.693327e-01,9.404314e-01,0.000000e+00
max,1.430584e+06,3.652450e+07,2.465727e+00,1.000000e+00,1.000000e+00


/tmp/ipykernel_3375/2235231115.py:19: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), negatives_per_user), random_state=0))


(213708, 5)

In [50]:
from catboost import CatBoostClassifier, Pool

# задаём имена колонок признаков и таргета
features = ['als_score', 'cnt_score']
target = 'target'

# Create the Pool object
train_data = Pool(
    data=candidates_for_train[features], 
    label=candidates_for_train[target])

# инициализируем модель CatBoostClassifier
cb_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.1,
    depth=6,
    loss_function='Logloss',
    verbose=100,
    random_seed=0
)

# тренируем модель
cb_model.fit(train_data)

0:	learn: 0.6490473	total: 67.4ms	remaining: 1m 7s
100:	learn: 0.5023899	total: 2.03s	remaining: 18.1s
200:	learn: 0.5015905	total: 4.12s	remaining: 16.4s
300:	learn: 0.5008853	total: 6.23s	remaining: 14.5s
400:	learn: 0.5002944	total: 8.34s	remaining: 12.5s
500:	learn: 0.4997685	total: 10.4s	remaining: 10.4s
600:	learn: 0.4992607	total: 12.5s	remaining: 8.33s
700:	learn: 0.4988429	total: 14.6s	remaining: 6.25s
800:	learn: 0.4984777	total: 16.8s	remaining: 4.16s
900:	learn: 0.4981234	total: 18.8s	remaining: 2.07s
999:	learn: 0.4977696	total: 20.9s	remaining: 0us


CatBoostClassifier(depth=6, iterations=1000, learning_rate=0.1, loss_function='Logloss', random_seed=0, verbose=100)

In [24]:
# загружаем рекомендации от двух базовых генераторов
als_recommendations_2 = pd.read_parquet("candidates/inference/als_recommendations.parquet")
content_recommendations_2 = pd.read_parquet("candidates/inference/content_recommendations.parquet")

candidates_to_rank = pd.merge(
    als_recommendations_2[["user_id", "item_id", "score"]].rename(columns={"score": "als_score"}),
    content_recommendations_2[["user_id", "item_id", "score"]].rename(columns={"score": "cnt_score"}),
    on=['user_id', 'item_id'],
    how="outer")

# оставляем только тех пользователей, что есть в тестовой выборке, для экономии ресурсов
candidates_to_rank = candidates_to_rank[candidates_to_rank["user_id"].isin(events_test_2["user_id"].drop_duplicates())]
print(len(candidates_to_rank))

14517152


In [53]:
inference_data = Pool(data=candidates_to_rank[features])
predictions = cb_model.predict_proba(inference_data)

candidates_to_rank["cb_score"] = predictions[:, 1]

# для каждого пользователя проставляем rank, начиная с 1 — это максимальный cb_score
candidates_to_rank = candidates_to_rank.sort_values(["user_id", "cb_score"], ascending=[True, False])
candidates_to_rank["rank"] = candidates_to_rank.groupby("user_id").cumcount() + 1

max_recommendations_per_user = 100
final_recommendations = candidates_to_rank.query("rank <= @max_recommendations_per_user")
print(len(final_recommendations))

7519400


In [58]:
events_inference = pd.concat([events_train, events_labels])

cb_events_recs_for_binary_metrics_5 = process_events_recs_for_binary_metrics(
    events_inference,
    events_test_2,  
    final_recommendations.rename(columns={"cb_score": "score"}), 
    top_k=5)

cb_precision_5, cb_recall_5 = compute_cls_metrics(cb_events_recs_for_binary_metrics_5)

print(f"precision: {cb_precision_5:.3f}, recall: {cb_recall_5:.3f}")

Common users: 75194
precision: 0.007, recall: 0.016


/tmp/ipykernel_2344/3370395532.py:31: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  events_recs_common["gt"] = events_recs_common["gt"].fillna(False)


# === Двухстадийный подход: построение признаков

In [25]:
import numpy as np
items["age"] = 2018 - items["publication_year"]
invalid_age_idx = items["age"] < 0
items.loc[invalid_age_idx, "age"] = np.nan
items["age"] = items["age"].astype("float")

candidates_for_train = candidates_for_train.merge(items[['item_id', 'age', 'average_rating']], on=['item_id'], how='left')
candidates_to_rank = candidates_to_rank.merge(items[['item_id', 'age', 'average_rating']], on=['item_id'], how='left')

In [26]:
display(candidates_to_rank)
print(candidates_to_rank['age'].median())

,user_id,item_id,als_score,cnt_score,age,average_rating
0,1000003,1232,0.484089,NaN,NaN,4.25
1,1000003,3636,0.579718,NaN,12.0,4.12
2,1000003,3985,NaN,0.924399,NaN,4.17
3,1000003,4214,0.376419,NaN,12.0,3.88
4,1000003,4588,0.411577,NaN,12.0,3.97
...,...,...,...,...,...,...
14517147,1430580,29844228,0.028825,0.947815,2.0,4.02
14517148,1430580,30226723,0.020854,NaN,1.0,4.02
14517149,1430580,32075671,NaN,0.947480,1.0,4.60
14517150,1430580,33795874,NaN,0.947473,1.0,3.51


7.0


In [28]:
display(events)

,user_id,item_id,started_at,read_at,is_read,rating,is_reviewed
0,1229132,22034,2015-07-12,2015-07-17,True,5,False
1,1229132,22318578,2015-06-07,2015-08-09,True,5,True
2,1229132,22551730,2015-06-24,2015-07-11,True,4,True
3,1229132,22816087,2015-09-27,2015-11-04,True,5,True
5,1229132,17910054,2015-03-04,2015-07-28,True,3,False
...,...,...,...,...,...,...,...
12914452,1364473,5297,2017-02-07,2017-02-26,True,5,False
12914453,1364473,4900,2016-12-22,2016-12-29,True,2,False
12914454,1364473,14836,2016-11-29,2017-01-15,True,3,False
12914456,1297020,10210,2012-06-05,2013-01-17,True,5,False


In [29]:
def get_user_features(events):
    """ считает пользовательские признаки """
    
    user_features = events.groupby("user_id").agg(
        reading_years=("started_at", lambda x: (x.max()-x.min()).days/365.25),
        books_read=('is_read', lambda x: sum([1 if i else 0 for i in x])),
        rating_avg=("rating", "mean"),
        rating_std=("rating", "std"))
    
    user_features["books_per_year"] = user_features["books_read"] / user_features["reading_years"]
    
    return user_features
    
user_features_for_train = get_user_features(events_train)
candidates_for_train = candidates_for_train.merge(user_features_for_train, on="user_id", how="left")
  
# оставим только тех пользователей, что есть в тесте, для экономии ресурсов
events_inference = pd.concat([events_train, events_labels])
events_inference = events_inference[events_inference["user_id"].isin(events_test["user_id"].drop_duplicates())]

user_features_for_ranking = get_user_features(events_inference)
candidates_to_rank = candidates_to_rank.merge(user_features_for_ranking, on="user_id", how="left")

In [30]:
print(candidates_for_train['books_read'].median())

32.0


In [31]:
# определяем индексы топ-10 жанров и всех остальных
genres_top_k = 10
genres_top_idx = genres.sort_values("votes", ascending=False).head(genres_top_k).index
genres_others_idx = list(set(genres.index) - set(genres_top_idx))

genres_top_columns = [f"genre_{id}" for id in genres_top_idx]
genres_others_column = "genre_others"
genre_columns =  genres_top_columns + [genres_others_column]

# составляем таблицу принадлежности книг к жанрам
item_genres = (
    pd.concat([
        # топ жанров
        pd.DataFrame(all_items_genres_csr[:, genres_top_idx].toarray(),columns=genres_top_columns),
        # все остальные жанры
        pd.DataFrame(all_items_genres_csr[:, genres_others_idx].sum(axis=1), columns=[genres_others_column])], 
        axis=1)
    .reset_index()
    .rename(columns={"index": "item_id_enc"})
)

# объединяем информацию принадлежности книг к жанрам с основной информацией о книгах
items = items.merge(item_genres, on="item_id_enc", how="left")

def get_user_genres(events, items, item_genre_columns):
    user_genres = (
        events
        .merge(items[["item_id"] + item_genre_columns], on="item_id", how="left")
        .groupby("user_id")[item_genre_columns].mean()
    )
    return user_genres
    
user_genres_for_train = get_user_genres(events_train, items, genre_columns)
candidates_for_train = candidates_for_train.merge(user_genres_for_train, on="user_id", how="left")

user_genres_for_ranking = get_user_genres(events_inference, items, genre_columns)
candidates_to_rank = candidates_to_rank.merge(user_genres_for_ranking, on="user_id", how="left")

In [32]:
romance_idx = genres.query("name == 'Romance'").index[0]

romance_col = f"genre_{romance_idx}"

round(candidates_for_train[romance_col].median(), 2)

0.04

In [33]:
from catboost import CatBoostClassifier, Pool

# задаём имена колонок признаков и таргета
features = ['als_score', 'cnt_score', 
    'age', 'average_rating', 'reading_years', 'books_read', 
    'rating_avg', 'rating_std', 
    'books_per_year'] + genre_columns
target = 'target'

# создаём Pool
train_data = Pool(
    data=candidates_for_train[features], 
    label=candidates_for_train[target])

# инициализируем модель CatBoostClassifier
cb_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.1,
    depth=6,
    loss_function='Logloss',
    verbose=100,
    random_seed=0,
)

# тренируем модель
cb_model.fit(train_data)

0:	learn: 0.6416189	total: 84ms	remaining: 1m 23s
100:	learn: 0.4525474	total: 2.82s	remaining: 25.1s
200:	learn: 0.4436218	total: 5.52s	remaining: 21.9s
300:	learn: 0.4368464	total: 8.24s	remaining: 19.1s
400:	learn: 0.4317846	total: 11s	remaining: 16.4s
500:	learn: 0.4270583	total: 13.7s	remaining: 13.7s
600:	learn: 0.4228040	total: 16.5s	remaining: 10.9s
700:	learn: 0.4191945	total: 19.2s	remaining: 8.18s
800:	learn: 0.4155880	total: 22s	remaining: 5.45s
900:	learn: 0.4122738	total: 24.7s	remaining: 2.72s
999:	learn: 0.4091125	total: 27.5s	remaining: 0us


CatBoostClassifier(depth=6, iterations=1000, learning_rate=0.1, loss_function='Logloss', random_seed=0, verbose=100)

In [34]:
inference_data = Pool(data=candidates_to_rank[features])
predictions = cb_model.predict_proba(inference_data)

candidates_to_rank["cb_score"] = predictions[:, 1]

# для каждого пользователя проставим rank, начиная с 1 — это максимальный cb_score
candidates_to_rank = candidates_to_rank.sort_values(["user_id", "cb_score"], ascending=[True, False])
candidates_to_rank["rank"] = candidates_to_rank.groupby("user_id").cumcount() + 1

max_recommendations_per_user = 100
final_recommendations = candidates_to_rank.query("rank <= @max_recommendations_per_user")

In [39]:
print(len(set(final_recommendations['user_id'])))

75194


In [40]:
final_recommendations.to_parquet('final_recommendations.parquet')

In [45]:
# для экономии ресурсов оставим события только тех пользователей, 
# для которых следует оценить рекомендации
events_inference = pd.concat([events_train, events_labels])
events_inference = events_inference[events_inference["user_id"].isin(events_test_2["user_id"].drop_duplicates())]

cb_events_recs_for_binary_metrics_5 = process_events_recs_for_binary_metrics(
    events_inference,
    events_test_2,
    final_recommendations.rename(columns={"cb_score": "score"}), 
    top_k=10)

cb_precision_5, cb_recall_5 = compute_cls_metrics(cb_events_recs_for_binary_metrics_5)

print(f"precision: {cb_precision_5:.3f}, recall: {cb_recall_5:.3f}")

Common users: 75194
precision: 0.011, recall: 0.058


/tmp/ipykernel_3375/3370395532.py:31: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  events_recs_common["gt"] = events_recs_common["gt"].fillna(False)


In [46]:
feature_importance = pd.DataFrame(cb_model.get_feature_importance(), 
    index=features, 
    columns=["fi"])
feature_importance = feature_importance.sort_values(
    "fi",
    ascending=False
)

print(feature_importance )

                       fi
als_score       29.425896
age             22.537467
average_rating  17.159766
books_read       2.899324
cnt_score        2.459789
genre_18         2.437989
reading_years    2.427590
genre_others     2.172915
genre_1          2.082951
genre_25         1.997304
genre_34         1.964973
genre_38         1.924919
books_per_year   1.805747
genre_24         1.462767
rating_avg       1.446812
genre_33         1.416937
genre_20         1.202162
genre_16         1.135885
genre_5          1.049570
rating_std       0.989237


In [7]:
import pandas as pd
final_recs = pd.read_parquet("final_recommendations.parquet")
items = pd.read_parquet("items.par")

In [9]:
items['item_id'].nunique()

43312

# === Онлайн-рекомендации

In [11]:
# получим энкодированные идентификаторы всех объектов, известных нам из events_train
train_item_ids_enc = events_train['item_id_enc'].unique()

max_similar_items = 10

# получаем списки похожих объектов, используя ранее полученную ALS-модель
# метод similar_items возвращает и сам объект, как наиболее похожий
# этот объект мы позже отфильтруем, но сейчас запросим на 1 больше
similar_items = als_model.similar_items(train_item_ids_enc, N=max_similar_items+1)

# преобразуем полученные списки в табличный формат
sim_item_item_ids_enc = similar_items[0]
sim_item_scores = similar_items[1]

similar_items = pd.DataFrame({
    "item_id_enc": train_item_ids_enc,
    "sim_item_id_enc": sim_item_item_ids_enc.tolist(), 
    "score": sim_item_scores.tolist()})
similar_items = similar_items.explode(
    ["sim_item_id_enc", "score"],
    ignore_index=True
)
# приводим типы данных
similar_items["sim_item_id_enc"] = similar_items["sim_item_id_enc"].astype("int")
similar_items["score"] = similar_items["score"].astype("float")

# получаем изначальные идентификаторы
similar_items["item_id_1"] = item_encoder.inverse_transform(similar_items["item_id_enc"])
similar_items["item_id_2"] = item_encoder.inverse_transform(similar_items["sim_item_id_enc"])
similar_items = similar_items.drop(columns=["item_id_enc", "sim_item_id_enc"])

# убираем пары с одинаковыми объектами
similar_items = similar_items.query("item_id_1 != item_id_2")

In [14]:
print(similar_items.query('item_id_1 == 7126'))

          score  item_id_1  item_id_2
25873  0.948725       7126       7190
25874  0.940997       7126      24280
25875  0.930144       7126       1953
25876  0.925066       7126      58696
25877  0.916340       7126      38296
25878  0.916015       7126       2932
25879  0.913951       7126       7184
25880  0.911433       7126     387749
25881  0.909872       7126       7733
25882  0.909454       7126      30597


In [15]:
similar_items.to_parquet("similar_items.parquet")

In [ ]:
def print_sim_items(item_id, similar_items):

    item_columns_to_use = ["item_id", "author", "title", "genre_and_votes", "average_rating", "ratings_count"]
    
    item_id_1 = items.query("item_id == @item_id")[item_columns_to_use]
    display(item_id_1)
    
    si = similar_items.query("item_id_1 == @item_id")
    si = si.merge(items[item_columns_to_use].set_index("item_id"), left_on="item_id_2", right_index=True)
    display(si)

print_sim_items(7144, similar_items)